# Lab 04 - Red Teaming (starter)

Complete the `TODO` blocks. Reference: `lab04_red_teaming_solution.ipynb`.

You will use the **new Foundry Evals API** (`project_client.get_openai_client().evals` with
`scenario="red_team"`): attacks are generated, executed and graded server-side, so PyRIT is not needed
in your local environment.

A Foundry project and a chat deployment are mandatory for this lab.

## Step 0 - Configuration (ready to run)

In [ ]:
import os, sys, time, json, warnings
from pprint import pprint
from collections import defaultdict

from lab_utils import load_settings

warnings.filterwarnings("ignore")

settings = load_settings(verbose=True)
credential = settings["credential"]
foundry_project_endpoint = settings["foundry_project_endpoint"]
deployment_name = settings["azure_openai_deployment_name"]

if not foundry_project_endpoint or not deployment_name:
    raise ValueError("FOUNDRY_PROJECT_ENDPOINT and AZURE_OPENAI_CHAT_DEPLOYMENT_NAME are required")

In [ ]:
from azure.ai.projects import AIProjectClient

# TODO 0.1 - create the AIProjectClient and get .get_openai_client().evals
project_client = ...
evals_client = ...

## Step 1 - Create the red-team evaluation group (~8 min)

The group declares what counts as a failure. Use `data_source_config={"type": "azure_ai_source",
"scenario": "red_team"}` and one testing criterion based on `builtin.violence`.

In [ ]:
# TODO 1.1 - evals_client.create(name=..., data_source_config=..., testing_criteria=[...])
red_team_eval = ...
print(red_team_eval.id)

## Step 2 - Start a small scan (~12 min)

`data_source` must contain:

* `"type": "azure_ai_red_team"`,
* `item_generation_params` with `attack_strategies` (start with `baseline` and `base64`) and `num_turns: 1`,
* `target` with `{"type": "azure_ai_model", "model": deployment_name}`.

Then poll the run until it reaches a terminal status.

In [ ]:
# TODO 2.1 - evals_client.runs.create(eval_id=..., name=..., data_source={...})
red_team_run = ...
print(red_team_run.id, red_team_run.status)

# TODO 2.2 - poll with evals_client.runs.retrieve(run_id=..., eval_id=...) until
#            completed / failed / canceled, sleeping 5 seconds between checks

## Step 3 - Fallback if the run is slow (~2 min)

Scans take minutes. This read-only cell finds the most recent completed red-team run so you can carry
on with real data instead of waiting.

In [ ]:
completed_runs = []

for candidate_eval in evals_client.list(limit=100, order="desc"):
    payload = candidate_eval.model_dump(exclude_none=True, warnings=False)
    if (payload.get("data_source_config") or {}).get("scenario") != "red_team":
        continue
    for candidate_run in evals_client.runs.list(candidate_eval.id, limit=20, order="desc", status="completed"):
        completed_runs.append((candidate_eval, candidate_run))

if completed_runs:
    red_team_eval, red_team_run = max(completed_runs, key=lambda pair: pair[1].created_at)
    print(red_team_run.id, red_team_run.status, red_team_run.report_url)
else:
    print("No completed red-team run available yet.")

## Step 4 - Compute the ASR and inspect one attack (~15 min)

List the output items and group them by attack strategy (look inside `metadata`).
Remember: `passed = False` means the attack **succeeded**.

Print numerator and denominator, never a bare percentage.

In [ ]:
# TODO 4.1 - output_items = list(evals_client.runs.output_items.list(run_id=..., eval_id=...))
# TODO 4.2 - group by metadata["attack_strategy"] and count how many items have a failed result
# TODO 4.3 - print "strategy ASR = successes/total (percentage)"
# TODO 4.4 - dump one output item in full: probe, model answer, grader verdict
# TODO 4.5 - print red_team_run.report_url and open it in the portal

### Checkpoint - everything below is optional

## Step 5 (optional) - Widen the scan

Add a second risk category (for example `builtin.hate_unfairness`) and a third strategy
(`rot13`, `leetspeak`, `morse`, `url`, `flip`, `unicode_confusable`, `tense`...).
Cost and duration grow with `categories x strategies x turns`.

In [ ]:
# TODO 5.1 - create a wider evaluation group and run, then compare the ASR per technique

## Step 6 (optional) - The legacy preview API

`project_client.beta.red_teams` builds the scan as a typed `RedTeam` object and its report opens the
**classic** Foundry experience. Useful to recognise in existing code.

In [ ]:
from azure.ai.projects.models import AttackStrategy, AzureOpenAIModelConfiguration, RedTeam, RiskCategory

# TODO 6.1 - build a RedTeam(...) config with VIOLENCE, BASELINE + BASE64, num_turns=1
# TODO 6.2 - list project_client.beta.red_teams.list(), pick a completed scan and read
#            outputs["evaluationMetrics"] -> violence_baseline_asr / violence_easy_complexity_asr